In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Inventory Management") \
    .getOrCreate()

print("Spark Session Created")

Spark Session Created


In [2]:
from google.colab import files
uploaded=files.upload()

Saving stock_movements.csv to stock_movements.csv


In [3]:
stock_df=spark.read.csv("stock_movements.csv",header=True,inferSchema=True)
stock_df.show()

+----------+------------+------------+-------------+--------+-------------+-------------+
|product_id|product_name|warehouse_id|movement_type|quantity|movement_date|reorder_level|
+----------+------------+------------+-------------+--------+-------------+-------------+
|       101|      Laptop|           1|           IN|      50|   2026-05-01|           10|
|       102|       Mouse|           2|           IN|      30|   2026-05-01|           20|
|       103|    Keyboard|           3|          OUT|      -5|   2026-05-02|           15|
|       104|     Monitor|           4|           IN|      20|   2026-05-02|            8|
|       105|     Printer|           5|          OUT|      -2|   2026-05-03|            5|
|       106|     Scanner|           6|           IN|      15|   2026-05-03|            6|
|       107|     Speaker|           7|          OUT|      -4|   2026-05-04|           12|
|       108|      Router|           8|           IN|      40|   2026-05-04|           10|
|       10

In [4]:
from pyspark.sql.functions import sum

total_stock_per_warehouse = stock_df.groupBy("warehouse_id").sum("quantity").alias("total_stock")
total_stock_per_warehouse.show()

+------------+-------------+
|warehouse_id|sum(quantity)|
+------------+-------------+
|           1|           50|
|           6|           15|
|           3|          -11|
|           5|           -2|
|           9|           -3|
|           4|           20|
|           8|           40|
|           7|           -4|
|          10|           60|
|           2|           55|
+------------+-------------+



In [ ]:
from p

In [5]:
from pyspark.sql.functions import col, when

current_stock = stock_df.groupBy("product_id", "warehouse_id").agg(sum("quantity").alias("current_stock"))

product_reorder_levels = stock_df.select("product_id", "reorder_level").distinct()

stock_status = current_stock.join(product_reorder_levels, on="product_id", how="left")


stock_status = stock_status.withColumn(
    "stock_category",
    when(col("current_stock") < col("reorder_level"), "Understock")
    .when(col("current_stock") > col("reorder_level") * 2, "Overstock")
    .otherwise("Optimal Stock")
)

stock_status.show()

+----------+------------+-------------+-------------+--------------+
|product_id|warehouse_id|current_stock|reorder_level|stock_category|
+----------+------------+-------------+-------------+--------------+
|       110|          10|           60|           25|     Overstock|
|       107|           7|           -4|           12|    Understock|
|       102|           2|           30|           20| Optimal Stock|
|       108|           8|           40|           10|     Overstock|
|       103|           3|           -5|           15|    Understock|
|       112|           3|           -6|           15|    Understock|
|       101|           1|           50|           10|     Overstock|
|       111|           2|           25|           12|     Overstock|
|       105|           5|           -2|            5|    Understock|
|       106|           6|           15|            6|     Overstock|
|       109|           9|           -3|            7|    Understock|
|       104|           4|         

In [6]:
stock_status.write.csv(
    "warehouse_output",
    header=True
)

This table shows the stock status for each product in each warehouse. An item is categorized as:
*   **Understock**: If `current_stock` is less than `reorder_level`.
*   **Overstock**: If `current_stock` is more than double the `reorder_level` (this threshold can be adjusted).
*   **Optimal Stock**: Otherwise.